# Phase 2: GDC Expression Analysis

Notebook này đọc output Phase 1, so sánh expression giữa Tumor và Normal, tính log2FC và p-value xấp xỉ theo Welch-style t-stat bằng Spark. Kết quả được ghi vào layer analysis trên HDFS.

Nguyên tắc chạy:
- Chỉ đọc `gdc_qc_pass_expression` từ layer analysis.
- Không đọc hoặc sửa raw/refined.
- Ghi output vào `hdfs://master11:9000/drugtarget/data/analysis/gdc_deg_result`.
- p-value là xấp xỉ normal từ Welch t-stat, phù hợp khi môi trường chưa có pandas/scipy/pyarrow.

In [1]:
import math
from pyspark.sql import SparkSession, functions as F, types as T

# Khai báo đường dẫn HDFS cho input Phase 1 và output Phase 2.
HDFS_BASE = "hdfs://master11:9000"
PHASE1_OUTPUT_PATH = f"{HDFS_BASE}/drugtarget/data/analysis/gdc_qc_pass_expression"
DEG_OUTPUT_PATH = f"{HDFS_BASE}/drugtarget/data/analysis/gdc_deg_result"

# Tạo SparkSession để xử lý expression theo gene trên dữ liệu đã qua QC.
spark = (
    SparkSession.builder.appName("drugtarget-gdc-phase2-expression-analysis")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .enableHiveSupport()
    .getOrCreate()
)

# Giảm log để notebook tập trung vào kết quả chính.
spark.sparkContext.setLogLevel("WARN")

print(f"Spark master: {spark.sparkContext.master}")
print(f"Input Phase 2: {PHASE1_OUTPUT_PATH}")
print(f"Output Phase 2: {DEG_OUTPUT_PATH}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/03 08:12:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark master: local[*]
Input Phase 2: hdfs://master11:9000/drugtarget/data/analysis/gdc_qc_pass_expression
Output Phase 2: hdfs://master11:9000/drugtarget/data/analysis/gdc_deg_result


In [2]:
def require_columns(frame, required_columns, frame_name: str) -> None:
    """Dừng pipeline nếu DataFrame thiếu cột bắt buộc."""
    missing_columns = sorted(set(required_columns) - set(frame.columns))
    if missing_columns:
        missing_text = ", ".join(missing_columns)
        raise ValueError(f"{frame_name} thiếu cột bắt buộc: {missing_text}")


# Đọc output Phase 1 làm input duy nhất cho phân tích DEG.
expr = spark.read.parquet(PHASE1_OUTPUT_PATH)

EXPR_REQUIRED_COLUMNS = [
    "sample_id",
    "case_id",
    "sample_group",
    "gene_id_base",
    "gene_name",
    "tpm",
    "log2_tpm",
]
require_columns(expr, EXPR_REQUIRED_COLUMNS, "gdc_qc_pass_expression")

print("Schema input Phase 2:")
expr.printSchema()
print("Số dòng và sample theo sample_group:")
expr.groupBy("sample_group").agg(
    F.count("*").alias("num_rows"),
    F.countDistinct("sample_id").alias("num_samples"),
    F.countDistinct("gene_id_base").alias("num_genes"),
).orderBy("sample_group").show(truncate=False)

Schema input Phase 2:
root
 |-- sample_id: string (nullable = true)
 |-- case_id: string (nullable = true)
 |-- sample_group: string (nullable = true)
 |-- gene_id_base: string (nullable = true)
 |-- gene_name: string (nullable = true)
 |-- tpm: double (nullable = true)
 |-- log2_tpm: double (nullable = true)

Số dòng và sample theo sample_group:


+------------+--------+-----------+---------+
|sample_group|num_rows|num_samples|num_genes|
+------------+--------+-----------+---------+
|normal      |1176696 |59         |19944    |
|tumor       |10570320|520        |19944    |
+------------+--------+-----------+---------+



In [3]:
# Chuẩn hóa nhãn nhóm để đảm bảo Tumor/Normal có cùng format khi so sánh.
expr2 = (
    expr.withColumn("sample_group_norm", F.lower(F.trim(F.col("sample_group"))))
    .withColumn(
        "group_label",
        F.when(F.col("sample_group_norm").contains("tumor"), F.lit("Tumor"))
        .when(F.col("sample_group_norm").contains("normal"), F.lit("Normal"))
        .otherwise(F.lit("Other")),
    )
    .filter(F.col("group_label").isin("Tumor", "Normal"))
    .select(
        "sample_id",
        "case_id",
        "gene_id_base",
        "gene_name",
        F.col("tpm").cast("double").alias("tpm"),
        F.col("log2_tpm").cast("double").alias("log2_tpm"),
        "group_label",
    )
)

# Kiểm tra dữ liệu có đủ hai nhóm để tính Tumor vs Normal.
group_counts = expr2.groupBy("group_label").agg(
    F.count("*").alias("num_rows"),
    F.countDistinct("sample_id").alias("num_samples"),
).orderBy("group_label")
print("Số dòng và sample sau khi chuẩn hóa group_label:")
group_counts.show(truncate=False)

actual_groups = {row["group_label"]: row["num_samples"] for row in group_counts.collect()}
missing_groups = sorted(set(["Tumor", "Normal"]) - set(actual_groups))
if missing_groups:
    raise ValueError(f"Thiếu nhóm bắt buộc để tính DEG: {missing_groups}")
if actual_groups["Tumor"] < 2 or actual_groups["Normal"] < 2:
    raise ValueError(f"Không đủ sample để tính Welch-style p-value: {actual_groups}")

Số dòng và sample sau khi chuẩn hóa group_label:


+-----------+--------+-----------+
|group_label|num_rows|num_samples|
+-----------+--------+-----------+
|Normal     |1176696 |59         |
|Tumor      |10570320|520        |
+-----------+--------+-----------+



In [4]:
@F.udf(T.DoubleType())
def normal_approx_p_value(t_stat):
    """Tính p-value hai phía xấp xỉ normal từ trị tuyệt đối của t-stat."""
    if t_stat is None:
        return None
    value = float(t_stat)
    if not math.isfinite(value):
        return None
    return float(math.erfc(abs(value) / math.sqrt(2.0)))


# Tính thống kê theo từng gene và từng nhóm Tumor/Normal.
gene_group_stats = (
    expr2.groupBy("gene_id_base", "gene_name", "group_label")
    .agg(
        F.avg("tpm").alias("mean_tpm"),
        F.avg("log2_tpm").alias("mean_log2_tpm"),
        F.variance("log2_tpm").alias("var_log2_tpm"),
        F.count("log2_tpm").cast("long").alias("n_values"),
        F.countDistinct("sample_id").cast("long").alias("num_samples"),
    )
)

# Đưa Tumor và Normal về cùng một dòng để tính log2FC và t-stat.
deg_stats = (
    gene_group_stats.groupBy("gene_id_base", "gene_name")
    .agg(
        F.max(F.when(F.col("group_label") == "Tumor", F.col("mean_tpm"))).alias("mean_tumor"),
        F.max(F.when(F.col("group_label") == "Normal", F.col("mean_tpm"))).alias("mean_normal"),
        F.max(F.when(F.col("group_label") == "Tumor", F.col("mean_log2_tpm"))).alias("mean_log2_tpm_tumor"),
        F.max(F.when(F.col("group_label") == "Normal", F.col("mean_log2_tpm"))).alias("mean_log2_tpm_normal"),
        F.max(F.when(F.col("group_label") == "Tumor", F.col("var_log2_tpm"))).alias("var_log2_tpm_tumor"),
        F.max(F.when(F.col("group_label") == "Normal", F.col("var_log2_tpm"))).alias("var_log2_tpm_normal"),
        F.max(F.when(F.col("group_label") == "Tumor", F.col("n_values"))).alias("n_tumor"),
        F.max(F.when(F.col("group_label") == "Normal", F.col("n_values"))).alias("n_normal"),
    )
    .filter(F.col("mean_log2_tpm_tumor").isNotNull() & F.col("mean_log2_tpm_normal").isNotNull())
    .withColumn("log2FC", F.col("mean_log2_tpm_tumor") - F.col("mean_log2_tpm_normal"))
    .withColumn(
        "welch_se",
        F.sqrt((F.col("var_log2_tpm_tumor") / F.col("n_tumor")) + (F.col("var_log2_tpm_normal") / F.col("n_normal"))),
    )
    .withColumn(
        "t_stat",
        F.when(
            (F.col("n_tumor") >= 2)
            & (F.col("n_normal") >= 2)
            & F.col("welch_se").isNotNull()
            & (F.col("welch_se") > 0),
            F.col("log2FC") / F.col("welch_se"),
        ),
    )
    .withColumn("p_value", normal_approx_p_value(F.col("t_stat")))
)

# Tạo cột hướng biểu hiện và cờ DEG theo ngưỡng đã chốt.
OUTPUT_COLUMNS = [
    "gene_id_base",
    "gene_name",
    "mean_tumor",
    "mean_normal",
    "mean_log2_tpm_tumor",
    "mean_log2_tpm_normal",
    "log2FC",
    "p_value",
    "deg_direction",
    "is_deg",
]

deg_result = (
    deg_stats.withColumn(
        "deg_direction",
        F.when(F.col("log2FC") > 0, F.lit("Upregulated"))
        .when(F.col("log2FC") < 0, F.lit("Downregulated"))
        .otherwise(F.lit("No change")),
    )
    .withColumn(
        "is_deg",
        F.coalesce((F.abs(F.col("log2FC")) >= F.lit(1.0)) & (F.col("p_value") < F.lit(0.05)), F.lit(False)),
    )
    .select(*OUTPUT_COLUMNS)
    .cache()
)

output_count = deg_result.count()
if output_count == 0:
    raise ValueError("Phase 2 không tạo được dòng DEG nào.")

print("Schema output Phase 2:")
deg_result.printSchema()
print(f"Số gene trong output Phase 2: {output_count}")

Schema output Phase 2:
root
 |-- gene_id_base: string (nullable = true)
 |-- gene_name: string (nullable = true)
 |-- mean_tumor: double (nullable = true)
 |-- mean_normal: double (nullable = true)
 |-- mean_log2_tpm_tumor: double (nullable = true)
 |-- mean_log2_tpm_normal: double (nullable = true)
 |-- log2FC: double (nullable = true)
 |-- p_value: double (nullable = true)
 |-- deg_direction: string (nullable = false)
 |-- is_deg: boolean (nullable = false)

Số gene trong output Phase 2: 19944


In [5]:
# Ghi kết quả DEG vào layer analysis, không ghi vào raw/refined.
deg_result.write.mode("overwrite").option("compression", "snappy").parquet(DEG_OUTPUT_PATH)
print(f"Đã ghi output Phase 2: {DEG_OUTPUT_PATH}")

Đã ghi output Phase 2: hdfs://master11:9000/drugtarget/data/analysis/gdc_deg_result


In [6]:
# Kiểm tra output path tồn tại trên HDFS sau khi ghi.
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
output_hdfs_path = spark._jvm.org.apache.hadoop.fs.Path(DEG_OUTPUT_PATH)
output_fs = output_hdfs_path.getFileSystem(hadoop_conf)
if not output_fs.exists(output_hdfs_path):
    raise AssertionError(f"Output Phase 2 chưa tồn tại trên HDFS: {DEG_OUTPUT_PATH}")

# Đọc lại output để xác nhận schema và các điều kiện chất lượng cơ bản.
deg_output = spark.read.parquet(DEG_OUTPUT_PATH)
if deg_output.columns != OUTPUT_COLUMNS:
    raise AssertionError(f"Schema Phase 2 không đúng: {deg_output.columns}")

bad_p_value_count = deg_output.filter(
    F.col("p_value").isNotNull() & ((F.col("p_value") < 0) | (F.col("p_value") > 1))
).count()
if bad_p_value_count != 0:
    raise AssertionError(f"Có {bad_p_value_count} dòng p_value ngoài khoảng [0, 1].")

expected_is_deg = F.coalesce((F.abs(F.col("log2FC")) >= F.lit(1.0)) & (F.col("p_value") < F.lit(0.05)), F.lit(False))
bad_is_deg_count = deg_output.filter(F.col("is_deg") != expected_is_deg).count()
if bad_is_deg_count != 0:
    raise AssertionError(f"Có {bad_is_deg_count} dòng is_deg không khớp ngưỡng DEG.")

print("Kiểm tra output Phase 2 hoàn tất.")
print("Đếm DEG theo is_deg và deg_direction:")
deg_output.groupBy("is_deg", "deg_direction").count().orderBy("is_deg", "deg_direction").show(truncate=False)

print("Top gene tăng trong Tumor:")
deg_output.filter(F.col("is_deg") == F.lit(True)).orderBy(F.desc("log2FC")).show(20, truncate=False)

print("Top gene giảm trong Tumor:")
deg_output.filter(F.col("is_deg") == F.lit(True)).orderBy(F.asc("log2FC")).show(20, truncate=False)

Kiểm tra output Phase 2 hoàn tất.
Đếm DEG theo is_deg và deg_direction:


+------+-------------+-----+
|is_deg|deg_direction|count|
+------+-------------+-----+
|false |Downregulated|6987 |
|false |No change    |441  |
|false |Upregulated  |9918 |
|true  |Downregulated|1692 |
|true  |Upregulated  |906  |
+------+-------------+-----+

Top gene tăng trong Tumor:


+---------------+---------+------------------+------------------+-------------------+--------------------+------------------+-----------------------+-------------+------+
|gene_id_base   |gene_name|mean_tumor        |mean_normal       |mean_log2_tpm_tumor|mean_log2_tpm_normal|log2FC            |p_value                |deg_direction|is_deg|
+---------------+---------+------------------+------------------+-------------------+--------------------+------------------+-----------------------+-------------+------+
|ENSG00000147689|FAM83A   |77.81623000000003 |0.7183779661016952|5.175194501263117  |0.686573360917304   |4.4886211403458125|0.0                    |Upregulated  |true  |
|ENSG00000118785|SPP1     |821.0397979245287 |26.89579491525423 |8.41813570058502   |3.9417878105934556  |4.476347889991565 |1.2788209301165663E-89 |Upregulated  |true  |
|ENSG00000143320|CRABP2   |587.6786654716983 |11.809174576271188|7.736909572412264  |3.37456699120864    |4.362342581203624 |3.9056761111867065E-

+---------------+---------+------------------+------------------+-------------------+--------------------+-------------------+-----------------------+-------------+------+
|gene_id_base   |gene_name|mean_tumor        |mean_normal       |mean_log2_tpm_tumor|mean_log2_tpm_normal|log2FC             |p_value                |deg_direction|is_deg|
+---------------+---------+------------------+------------------+-------------------+--------------------+-------------------+-----------------------+-------------+------+
|ENSG00000168484|SFTPC    |1062.6113694339617|27325.21958474577 |6.478240496351828  |14.669160354203495  |-8.190919857851668 |0.0                    |Downregulated|true  |
|ENSG00000204305|AGER     |122.22797886792453|3829.740664406779 |5.199981433581533  |11.766974797206116  |-6.566993363624583 |0.0                    |Downregulated|true  |
|ENSG00000066405|CLDN18   |83.94318528301886 |1011.7985559322034|3.566415464370427  |9.890677417828234   |-6.324261953457807 |0.0           

In [7]:
# Giải phóng cache sau khi hoàn tất Phase 2.
deg_result.unpersist()
print("Hoàn tất Phase 2: GDC Expression Analysis")

Hoàn tất Phase 2: GDC Expression Analysis
